# Stage 2: Shortlisting with BM25 + Keyword Overlap

The evidence corpus contains ~1.2 million passages but most are irrelevant to any given claim. This notebook narrows that pool down to a manageable shortlist for each claim by combining two cheap signals:

1. **BM25** — a sparse lexical retriever, indexed once over the cleaned BM25 view of the corpus.
2. **Keyword overlap** — a re-ranking signal that counts how many of the claim's named entities, noun chunks, and content tokens appear inside each candidate.

The two scores are combined as `z(bm25) + α · keyword_overlap`. The mixing weight α is tuned on the dev set by sweeping a grid and picking the value that maximises recall@1000.

The output is a top-5000 shortlist for every claim in `./shortlists/`, which the neural rerankers in Stage 3 read directly.

## 1. Setup

Import `bm25s` for the BM25 index, `spaCy` for keyword extraction, and configure the key hyperparameters: shortlist size, BM25 pool size, and BM25's `k1` and `b` parameters.

In [ ]:
import sys

sys.path.insert(0, "../src")  # repo-local package
import json
import os

import bm25s
import numpy as np
from factcheck import config as cfg
from factcheck.config import load_json as load
from factcheck.keywords import extract_keywords, fuse, keyword_overlap
from factcheck.metrics import recall_at
from tqdm.notebook import tqdm

In [ ]:
PROC_DIR = cfg.PROC_DIR
OUT_DIR = cfg.SL_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Key hyperparameters --------------------------------------------
TOP_M = 5000  # final shortlist size (was 1000)
BM25_POOL = 10000  # candidates from BM25 before NER reranks them
BM25_K1 = 1.2  # tuned slightly down from 1.5 (shorter passages)
BM25_B = 0.75
# NER_BOOST_ALPHA is tuned automatically on dev below

## 2. Load preprocessed data

Load the three cleaned views produced by Stage 1 (BM25 view of evidence, keyword view of evidence, and the augmented claim files for train / dev / test). The evidence IDs and their cleaned texts are turned into parallel lists so they can be passed directly to the BM25 indexer.

In [ ]:
evidence_bm25 = load(PROC_DIR / "evidence_bm25.json")
evidence_kw = load(PROC_DIR / "evidence_keyword.json")
train_claims = load(PROC_DIR / "train_claims_proc.json")
dev_claims = load(PROC_DIR / "dev_claims_proc.json")
test_claims = load(PROC_DIR / "test_claims_proc.json")

evidence_ids = list(evidence_bm25.keys())
evidence_texts = [evidence_bm25[e] for e in evidence_ids]
evidence_kw_l = [evidence_kw[e] for e in evidence_ids]
print(f"corpus: {len(evidence_ids):,}")

## 3. Build the BM25 index

Tokenise the corpus with English stop-word removal and a Snowball stemmer (so morphological variants like `warmed` / `warming` / `warmer` collapse to a single token), then build the BM25 index. The stemmer is wrapped in a try/except in case the installed `bm25s` version doesn't ship with one, in which case we fall back to no stemming.

In [ ]:
# bm25s exposes a built-in stemmer via `stemmer="snowball"` in newer versions; if your
# version doesn't support it, fall back to None and we rely on the heavier corpus.
try:
    print("Tokenising corpus with snowball stemmer...")
    corpus_tokens = bm25s.tokenize(evidence_texts, stopwords="en", stemmer="snowball", show_progress=True)
    STEMMER = "snowball"
except Exception as e:
    print(f"  snowball not available ({e}); falling back to no stemmer.")
    corpus_tokens = bm25s.tokenize(evidence_texts, stopwords="en", stemmer=None, show_progress=True)
    STEMMER = None

print("Building BM25 index...")
bm25 = bm25s.BM25(k1=BM25_K1, b=BM25_B)
bm25.index(corpus_tokens, show_progress=True)
print("done.")

## 4. Keyword extractor

Each claim is run through spaCy and yields two sets of strings: a tight set of **named entities** and a broader set of **all content keywords** (entities + noun chunks + individual NOUN/PROPN/ADJ/NUM tokens longer than two characters). Every string is normalised to lowercase, alphanumeric form so it matches the `keyword` view of the evidence corpus produced in Stage 1.

The two sets are kept separate so that the scoring function can weight entities more heavily than ordinary keywords.

In [ ]:
# Sanity check
demo = list(train_claims.values())[0]["claim_text"]
ents, kws = extract_keywords(demo)
print("claim   :", demo)
print("entities:", ents)
print("keywords:", kws)

## 5. Scoring function

Three helpers:

- `_zscore(x)` — standardises an array. We use z-scores rather than min-max normalisation because at a pool size of 10 000 the BM25 tail is essentially flat near zero, which would squash the head of the distribution and let the much smaller overlap signal dominate artificially.
- `_keyword_overlap(ev, ents, kws)` — sums weighted matches of each claim keyword inside an evidence's keyword view. Named entities count for 2.0 and ordinary keywords for 1.0, with the result divided by the total possible weight so the score lies in roughly [0, 1].
- `shortlist_one(...)` — retrieves the top `BM25_POOL = 10 000` candidates from BM25, computes the overlap signal, combines the two as `z(bm25) + α · overlap`, and returns the top `TOP_M = 5 000`.

In [ ]:
def shortlist_one(claim_bm25_text, ents, kws, top_m=TOP_M, alpha=0.5):
    """BM25 top-BM25_POOL, re-scored by z(bm25) + alpha * keyword overlap, keep top_m."""
    q = bm25s.tokenize(claim_bm25_text, stopwords="en", stemmer=STEMMER, show_progress=False)
    doc_ids, scores = bm25.retrieve(q, k=BM25_POOL, show_progress=False)
    doc_ids, scores = doc_ids[0], scores[0]
    if kws:
        overlap = [keyword_overlap(evidence_kw_l[di], ents, kws) for di in doc_ids]
        combined = fuse(scores, overlap, alpha)
    else:
        combined = scores
    order = np.argsort(-combined)[:top_m]
    return [evidence_ids[doc_ids[i]] for i in order]

## 6. Pre-extract keywords for every claim

spaCy is the slowest step in this notebook, so we run it once per claim and cache the results. This way the α-tuning sweep below can reuse the same keyword sets across all values of α.

In [ ]:
def cache_keywords(claims):
    out = {}
    for cid, c in tqdm(claims.items(), desc="spaCy"):
        out[cid] = extract_keywords(c["claim_text"])
    return out


train_kw = cache_keywords(train_claims)
dev_kw = cache_keywords(dev_claims)
test_kw = cache_keywords(test_claims)

## 7. Tune the keyword weight α on dev

Build a shortlist for every dev claim at several values of α (from 0 — pure BM25 — up to 1.2 — keyword overlap dominates), compute recall at four cut-off sizes, and pick the α that maximises recall@1000. recall@1000 is the relevant operating point because the bi-encoder in Stage 3 reranks the top-1000 region of the shortlist.

In [ ]:
def build_shortlists(claims, kw_cache, alpha, top_m=TOP_M):
    out = {}
    for cid, c in tqdm(claims.items(), desc=f"alpha={alpha:.2f}"):
        ents, kws = kw_cache[cid]
        out[cid] = shortlist_one(c["claim_bm25"], ents, kws, top_m=top_m, alpha=alpha)
    return out


alpha_grid = [0.2, 0.4, 0.6, 0.8, 1.2, 1.4, 1.6]
best_alpha, best_r = None, -1.0
print(f"{'alpha':<6s} | {'r@100':>6s} {'r@500':>6s} {'r@1000':>6s} {'r@5000':>6s}")
for a in alpha_grid:
    sl = build_shortlists(dev_claims, dev_kw, a, top_m=TOP_M)
    r100 = recall_at(sl, dev_claims, 100)
    r500 = recall_at(sl, dev_claims, 500)
    r1000 = recall_at(sl, dev_claims, 1000)
    r5000 = recall_at(sl, dev_claims, 5000)
    print(f"{a:<6.2f} | {r100:>6.4f} {r500:>6.4f} {r1000:>6.4f} {r5000:>6.4f}")
    # we maximise recall@1000 since that's where downstream rerankers operate
    if r1000 > best_r:
        best_r, best_alpha = r1000, a
print(f"\n>>> Best alpha by recall@1000 = {best_alpha} (recall {best_r:.4f})")

## 8. Build final shortlists with the tuned α

Re-run `shortlist_one` for every claim in train / dev / test using the α that won the sweep above, then print recall at several cut-offs as a sanity check.

In [ ]:
ALPHA = best_alpha
train_sl = build_shortlists(train_claims, train_kw, ALPHA)
dev_sl = build_shortlists(dev_claims, dev_kw, ALPHA)
test_sl = build_shortlists(test_claims, test_kw, ALPHA)

In [ ]:
print(f"Dev shortlist recall (alpha={ALPHA}):")
for k in (100, 500, 1000, 5000):
    print(f"  r@{k:<5d} = {recall_at(dev_sl, dev_claims, k):.4f}")

print("\nTrain shortlist recall:")
for k in (100, 500, 1000, 5000):
    print(f"  r@{k:<5d} = {recall_at(train_sl, train_claims, k):.4f}")

## 9. Save shortlists

Write the three shortlists to disk along with a small config file recording the hyperparameters used. Stage 3 reads `<split>_shortlist.json` directly.

In [ ]:
for name, sl in [("train", train_sl), ("dev", dev_sl), ("test", test_sl)]:
    p = OUT_DIR / f"{name}_shortlist.json"
    with open(p, "w", encoding="utf-8") as f:
        json.dump(sl, f)
    print(f"wrote {p} ({os.path.getsize(p) / 1e6:.1f} MB)")

# Save the tuned alpha and config for later stages
with open(OUT_DIR / "shortlist_config.json", "w") as f:
    json.dump(
        {
            "alpha": ALPHA,
            "top_m": TOP_M,
            "bm25_pool": BM25_POOL,
            "k1": BM25_K1,
            "b": BM25_B,
            "stemmer": STEMMER,
        },
        f,
        indent=2,
    )

## Summary

The notebook produces three JSON files in `./shortlists/`, each mapping a claim ID to its ranked list of 5000 evidence IDs:

- `train_shortlist.json`, `dev_shortlist.json`, `test_shortlist.json`

Plus `shortlist_config.json` which records the tuned α, the BM25 hyperparameters, and which stemmer was used. The recall numbers printed above bound everything that follows: nothing downstream can retrieve evidence that isn't in this shortlist.